# 03 · Exact and lexical retrieval baselines

## The engineering question
What can a simple exhaustive vector search and a lexical retriever already solve?

**Learning outcomes:** implement cosine ranking directly; inspect BM25's exact-token behavior; establish a baseline before approximation or fusion; keep tuning and test queries separate.

Exact search scores every eligible document. Its cost grows approximately with corpus size times embedding dimension, but for small corpora that can be simpler than building an ANN index. BM25 rewards matching query terms using inverse-document frequency and document-length normalization. It can preserve distinctions such as error codes that semantic encoders blur. Neither score scale is a relevance probability, so raw scores should not be added blindly.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("03", "B", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from vector_masterclass.data import documents, queries, validate_gold
from vector_masterclass.models import embed
from vector_masterclass.retrieval import exact_search, LexicalIndex
from vector_masterclass.metrics import ranking_metrics

assert validate_gold()["human_review"] == "approved", (
    "Gold labels changed; review before model comparison"
)
docs = documents(tenant="public")
ids = [d.key for d in docs]
matrix = embed([d.text for d in docs])
lexical = LexicalIndex([d.text for d in docs], ids)
rows = []
for q in queries("development"):
    if not q.answerable:
        continue
    dense = exact_search(embed([q.text], role="query")[0], matrix, ids, k=3)
    sparse = lexical.search(q.text, k=3)
    for method, hits in [("dense_exact", dense), ("BM25", sparse)]:
        metrics = ranking_metrics([id_ for id_, score in hits], q.relevance, 3)
        rows.append({"query": q.id, "method": method, "top_document": hits[0][0], **metrics})
results = lesson.table(pd.DataFrame(rows))
display(results[["query", "method", "top_document", "recall", "mrr"]])
fig, ax = plt.subplots()
results.pivot(index="query", columns="method", values="mrr").plot.bar(
    ax=ax, color=["#2563eb", "#f59e0b"]
)
ax.set(
    ylabel="Reciprocal rank at 3",
    ylim=(0, 1.15),
    title="Development queries: inspect errors before adding complexity",
)
ax.legend(title="Retriever", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Failure and interpretation
Both retrievers return candidates even for an unsupported question. Ranking alone does not establish that the corpus contains an answer. A later lesson calibrates abstention separately. If both methods solve this tiny set, that is a reason to expand the evaluation, not proof that a production problem is solved.

**Exercise:** inspect the Atlas E104/E140 results. Change a query to an exact identifier, then to a paraphrase. Explain the difference using term matching and semantic similarity. Do not alter the frozen test queries to make a preferred method win.

Prefer exhaustive search when its measured latency fits a small workload. Prefer lexical lookup for exact codes or names, and measure hybrid retrieval for mixed queries. ANN is an efficiency mechanism; it cannot repair a poor embedding model.

In [ ]:
unsupported = queries("development")[-1]
assert not unsupported.answerable
print("Unsupported query still gets lexical candidates:", lexical.search(unsupported.text, 2))
manifest = lesson.finish(
    {
        "gold_approved": validate_gold()["human_review"] == "approved",
        "aligned_vectors": matrix.shape[0] == len(ids),
        "both_baselines_executed": set(results.method) == {"dense_exact", "BM25"},
        "development_only": set(results["query"]).issubset({q.id for q in queries("development")}),
    },
    observations={
        "baseline_means": results.groupby("method")[["recall", "mrr", "ndcg"]].mean().to_dict()
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.